# Avengerssssssss.... "ENSEMBLE"

In [1]:
print("An ensemble of XGBoost and Random Forest!")

An ensemble of XGBoost and Random Forest!


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_log_error

# Load the training data (replace 'train.csv' with your actual file path)
train_df = pd.read_csv('train.csv', index_col='id')

# Encode 'Sex' column (1 for male, 2 for female)
train_df['Sex'] = train_df['Sex'].map({'male': 1, 'female': 2})

# Ensure numeric columns
numeric_cols = ['Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp', 'Calories']
train_df[numeric_cols] = train_df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Handle missing values (use training set means)
train_means = train_df.mean()
train_df = train_df.fillna(train_means)

# Feature engineering (keep high-importance features)
train_df['Duration_Heart_Rate'] = train_df['Duration'] * train_df['Heart_Rate']
train_df['Duration_Squared'] = train_df['Duration'] ** 2
train_df['Heart_Rate_Squared'] = train_df['Heart_Rate'] ** 2
train_df['Age_BMI'] = train_df['Age'] * (train_df['Weight'] / (train_df['Height'] / 100) ** 2)
train_df['Heart_Rate_Body_Temp'] = train_df['Heart_Rate'] * train_df['Body_Temp']

# Transform target to log(Calories + 1) for RMSLE
train_df['log_Calories'] = np.log1p(train_df['Calories'])

# Features (drop low-importance: BMI, log_Duration, Body_Temp, Height)
features = ['Sex', 'Age', 'Weight', 'Duration', 'Heart_Rate', 
            'Duration_Heart_Rate', 'Duration_Squared', 'Heart_Rate_Squared', 
            'Age_BMI', 'Heart_Rate_Body_Temp']
X_train = train_df[features]
y_train = train_df['log_Calories']

# Train XGBoost model with best parameters
xgb_model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, 
                         subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb_model.fit(X_train, y_train)

# Train Random Forest model
rf_model = RandomForestRegressor(n_estimators=200, max_depth=15, 
                                 min_samples_split=5, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate ensemble on training data
xgb_pred_log = xgb_model.predict(X_train)
rf_pred_log = rf_model.predict(X_train)
ensemble_pred_log = 0.6 * xgb_pred_log + 0.4 * rf_pred_log
ensemble_pred = np.expm1(ensemble_pred_log)
train_rmsle = np.sqrt(mean_squared_log_error(train_df['Calories'], np.maximum(ensemble_pred, 0)))
print(f"Training RMSLE (Ensemble): {train_rmsle:.4f}")

# Cross-validation for XGBoost (for reference)
from sklearn.model_selection import cross_val_score
xgb_scores = cross_val_score(xgb_model, X_train, y_train, cv=5, 
                             scoring='neg_mean_squared_log_error')
print(f"XGBoost Cross-Validation RMSLE: {np.sqrt(-xgb_scores.mean()):.4f}")

# Cross-validation for Random Forest
rf_scores = cross_val_score(rf_model, X_train, y_train, cv=5, 
                            scoring='neg_mean_squared_log_error')
print(f"Random Forest Cross-Validation RMSLE: {np.sqrt(-rf_scores.mean()):.4f}")

# Load and preprocess test data (replace 'test.csv' with your actual file path)
test_df = pd.read_csv('test.csv', index_col='id')
test_df['Sex'] = test_df['Sex'].map({'male': 1, 'female': 2})
test_df[numeric_cols[:-1]] = test_df[numeric_cols[:-1]].apply(pd.to_numeric, errors='coerce')
test_df = test_df.fillna(train_means)

# Feature engineering for test data
test_df['Duration_Heart_Rate'] = test_df['Duration'] * test_df['Heart_Rate']
test_df['Duration_Squared'] = test_df['Duration'] ** 2
test_df['Heart_Rate_Squared'] = test_df['Heart_Rate'] ** 2
test_df['Age_BMI'] = test_df['Age'] * (test_df['Weight'] / (test_df['Height'] / 100) ** 2)
test_df['Heart_Rate_Body_Temp'] = test_df['Heart_Rate'] * test_df['Body_Temp']

# Features for prediction
X_test = test_df[features]

# Make ensemble predictions
xgb_pred_log = xgb_model.predict(X_test)
rf_pred_log = rf_model.predict(X_test)
ensemble_pred_log = 0.6 * xgb_pred_log + 0.4 * rf_pred_log
predictions = np.expm1(ensemble_pred_log)

# Clip predictions to 5th–95th percentile of training Calories
calories_bounds = train_df['Calories'].quantile([0.05, 0.95]).values
predictions = np.clip(predictions, calories_bounds[0], calories_bounds[1])

# Create submission DataFrame
submission_df = pd.DataFrame({
    'id': test_df.index,
    'Calories': predictions.round(3)
})

# Save to CSV
submission_df.to_csv('submission_ensemble.csv', index=False)

print("Submission file 'submission_ensemble.csv' created successfully!")

Training RMSLE (Ensemble): 0.0537
XGBoost Cross-Validation RMSLE: 0.0174
